# Sigma2 Stability: Real Experiment Sequences

Companion to `2026-02-19_sigma2-stability.ipynb`. Replaces generated toy
experiments with **real multi-ISI sequences** from `exp_list`.
Each real sequence contains ~5 trials per ISI; we subsample N sequences
per MC rep to mirror the actual fitting scenario.

| Section | Question |
|---------|----------|
| A (toy) | Per-ISI stability with toy exps — baseline reference |
| B (real) | Stability using N real sequences per rep (N = 1, 5, 10, 36) |
| C (real) | ISI subset informativeness with real sequences |

In [ ]:
import sys, os, yaml, torch, random
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from collections import defaultdict
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from pathlib import Path
from scipy.spatial.distance import pdist
from tqdm.notebook import trange, tqdm

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path
from utls.plotting import ensure_dir
from utls.loading import (load_results_with_exclusion_2, move_sequences_to_used,
                           load_results_with_exclusion_no_dropping)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from encoders import *
from utls.toy_experiments import (
    make_isi_n_block_experiment, make_toy_experiment_list, make_multi_isi_toy_experiments,
)
from utls.sigma_fitting import log_mid, make_grid, auc_to_dprime

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

def median_pairwise_distance(X, metric='euclidean', n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))

CONFIG_PATH = (
    '/om2/user/bjmedina/auditory-memory/memory/'
    'model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml'
)
model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
exp_cfg    = model_cfg['experiment']
which_task = exp_cfg['which_task']
is_multi   = exp_cfg['is_multi']
which_isi  = exp_cfg.get('which_isi', None)
isis       = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]
metric     = model_cfg['metric']
noise_cfg  = model_cfg['noise_model']
noise_mode = noise_cfg['name']
t_step     = noise_cfg['t_step']
repr_cfg   = model_cfg['representation']
time_avg   = repr_cfg['time_avg']
encoder_type = repr_cfg['type']
layer      = repr_cfg.get('layer', None)
pc_dims    = repr_cfg.get('pc_dims', None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve  = compute_human_curve(human_runs, is_multi, which_isi)
time_avg_tag = 'time_avg' if time_avg else 'nontime_avg'
print('ISIs:', isis)
print('Human d\':', human_curve)
print(f'Total real sequences: {len(exp_list)}')

In [ ]:
NN_ENCODERS  = {'kell2018', 'resnet50'}
encoder_task = 'word_speaker_audioset' if encoder_type in NN_ENCODERS else 'audioset'
encoder_cfg  = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device='cuda',
)
if encoder_type in NN_ENCODERS: encoder_cfg['layer'] = layer
if encoder_type == 'texture':   encoder_cfg['pc_dims'] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
encoder      = build_encoder(encoder_cfg)
X            = encode_stimuli(encoder, all_files)
X_np         = X.detach().cpu().numpy()
print('Shape:', X_np.shape, ' NaN?', torch.isnan(X).any().item())

d50 = median_pairwise_distance(X_np, metric='cosine')
print(f'd50 = {d50:.6f}')

param_bounds = {
    'sigma0': (noise_cfg['sigma0_min'],         noise_cfg['sigma0_max']),
    'sigma1': (noise_cfg['sigma1_min'] * d50,   noise_cfg['sigma1_max'] * d50),
    'sigma2': (noise_cfg['sigma2_min'] * d50,   noise_cfg['sigma2_max'] * d50),
}
for k, v in param_bounds.items():
    print(f'  {k}: ({v[0]:.6f}, {v[1]:.6f})')

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f'Stimulus pool: {len(stimulus_pool)}')

In [ ]:
sigma0_fixed = log_mid(*param_bounds['sigma0'])
sigma1_fixed = log_mid(*param_bounds['sigma1'])
print(f'Fixed sigma0 = {sigma0_fixed:.6f}')
print(f'Fixed sigma1 = {sigma1_fixed:.6f}')

isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}
N_MC      = 64
N_PER_DIM = 8

sigma2_grid = make_grid(param_bounds['sigma2'][0], param_bounds['sigma2'][1],
                        N_PER_DIM, spacing='log')
print('sigma2 grid:', [f'{v:.6f}' for v in sigma2_grid])

print('\nHuman d\' targets:')
for isi_val in [8, 16, 32, 64]:
    idx = isi_to_hc_idx.get(isi_val)
    if idx is not None:
        print(f'  ISI {isi_val}: {human_curve[idx]:.4f}')
    else:
        print(f'  ISI {isi_val}: NOT in isis list!')

In [ ]:
def run_sigma_sweep(sigma_name, sigma_grid, fixed_sigmas, exps_by_isi,
                    isi_to_hc_idx, human_curve, N_MC, t_step, noise_mode,
                    metric, X, name_to_idx, base_seed=0):
    """Sweep one sigma over sigma_grid; for each value run N_MC reps.
    Returns list of dicts with sigma_value, mse_mean, mse_std, dprime_mean, dprime_std.
    """
    results = []
    for sig_idx, sigma_val in enumerate(sigma_grid):
        sigmas = dict(fixed_sigmas)
        sigmas[sigma_name] = sigma_val
        mse_per_rep    = []
        dprime_per_rep = []

        for rep in trange(N_MC, desc=f'{sigma_name}={sigma_val:.4g}', leave=False):
            rep_mse = []
            rep_dp  = []
            for isi_val, exps in exps_by_isi.items():
                if not exps: continue
                hc_idx   = isi_to_hc_idx.get(isi_val)
                if hc_idx is None: continue
                human_dp = human_curve[hc_idx]
                run_out  = run_experiment_scores(
                    sigma0=sigmas['sigma0'], sigma1=sigmas['sigma1'], sigma2=sigmas['sigma2'],
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X, name_to_idx=name_to_idx,
                    experiment_list=exps, debug=False,
                    seed=base_seed + isi_val*1_000_000 + sig_idx*10_000 + rep,
                )
                hits = np.asarray(run_out['hits']); fas = np.asarray(run_out['fas'])
                if len(hits)==0 or len(fas)==0: continue
                y = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
                dp = auc_to_dprime(roc_auc_score(y, -np.concatenate([hits,fas])))
                rep_mse.append((dp - human_dp)**2)
                rep_dp.append(dp)
            if rep_mse:
                mse_per_rep.append(np.mean(rep_mse))
                dprime_per_rep.append(np.mean(rep_dp))

        results.append({
            'sigma_value':  sigma_val,
            'mse_mean':     np.mean(mse_per_rep)    if mse_per_rep    else np.nan,
            'mse_std':      np.std(mse_per_rep)     if mse_per_rep    else np.nan,
            'dprime_mean':  np.mean(dprime_per_rep) if dprime_per_rep else np.nan,
            'dprime_std':   np.std(dprime_per_rep)  if dprime_per_rep else np.nan,
        })
    return results

In [ ]:
def infer_trial_isis(sequence):
    """Return the ISI for each repeat (target) trial in a sequence, in trial order.

    Assumes each stimulus appears at most twice.
    ISI = repeat_position - first_position - 1.
    Returns list[int] with one entry per repeat trial.
    First-presentation (foil) trials are not included.
    """
    first_seen = {}
    trial_isis = []
    for i, stim in enumerate(sequence):
        if stim in first_seen:
            trial_isis.append(i - first_seen[stim] - 1)
        else:
            first_seen[stim] = i
    return trial_isis

# Sanity check on first real sequence
example_isis = infer_trial_isis(exp_list[0])
print(f'Sequence 0 length: {len(exp_list[0])} stimuli')
print(f'Trial ISIs: {example_isis}')
from collections import Counter
print(f'ISI counts: {dict(Counter(example_isis))}')

In [ ]:
def run_real_sweep(sigma_name, sigma_grid, fixed_sigmas, real_exp_list, n_seqs_per_rep,
                   target_isis, isi_to_hc_idx, human_curve,
                   N_MC, t_step, noise_mode, metric, X, name_to_idx, base_seed=0):
    """Sweep sigma using real multi-ISI sequences.

    For each MC rep:
      - Sample n_seqs_per_rep sequences from real_exp_list (without replacement)
      - Run run_experiment_scores on each sequence individually
      - Split hits by ISI using infer_trial_isis (assumes hits returned in trial order)
      - Use all FAs from each sequence as the false-alarm pool for every ISI split
      - Compute d' per target ISI, average MSE across ISIs

    Note: run_experiment_scores is assumed to return hits in the same order as
    repeat trials appear in the sequence. Mismatches are silently skipped.
    """
    seq_trial_isis = [infer_trial_isis(seq) for seq in real_exp_list]
    results = []
    for sig_idx, sigma_val in enumerate(sigma_grid):
        sigmas = dict(fixed_sigmas)
        sigmas[sigma_name] = sigma_val
        mse_per_rep, dprime_per_rep = [], []

        for rep in trange(N_MC, desc=f'{sigma_name}={sigma_val:.4g}', leave=False):
            rng = np.random.default_rng(base_seed + sig_idx * 10_000 + rep)
            n_sample = min(n_seqs_per_rep, len(real_exp_list))
            seq_indices = rng.choice(len(real_exp_list), size=n_sample, replace=False)

            hits_list, isi_list, fas_list = [], [], []
            for si in seq_indices:
                seq      = real_exp_list[si]
                t_isis   = seq_trial_isis[si]
                run_out  = run_experiment_scores(
                    sigma0=sigmas['sigma0'], sigma1=sigmas['sigma1'], sigma2=sigmas['sigma2'],
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X, name_to_idx=name_to_idx,
                    experiment_list=[seq], debug=False,
                    seed=base_seed + sig_idx * 10_000 + rep * 1000 + int(si),
                )
                h = np.asarray(run_out['hits'])
                f = np.asarray(run_out['fas'])
                # ordering check: hits should match repeat trial count
                if len(h) != len(t_isis):
                    continue
                hits_list.append(h)
                isi_list.extend(t_isis)
                fas_list.append(f)

            if not hits_list:
                continue
            all_hits = np.concatenate(hits_list)
            all_isis = np.array(isi_list)
            all_fas  = np.concatenate(fas_list) if fas_list else np.array([])
            if len(all_fas) == 0:
                continue

            rep_mse, rep_dp = [], []
            for isi_val in target_isis:
                mask     = all_isis == isi_val
                hits_isi = all_hits[mask]
                if len(hits_isi) == 0:
                    continue
                hc_idx = isi_to_hc_idx.get(isi_val)
                if hc_idx is None:
                    continue
                human_dp = human_curve[hc_idx]
                y  = np.concatenate([np.ones(len(hits_isi)), np.zeros(len(all_fas))])
                dp = auc_to_dprime(roc_auc_score(y, -np.concatenate([hits_isi, all_fas])))
                rep_mse.append((dp - human_dp) ** 2)
                rep_dp.append(dp)
            if rep_mse:
                mse_per_rep.append(np.mean(rep_mse))
                dprime_per_rep.append(np.mean(rep_dp))

        results.append({
            'sigma_value':  sigma_val,
            'mse_mean':     np.mean(mse_per_rep)    if mse_per_rep    else np.nan,
            'mse_std':      np.std(mse_per_rep)     if mse_per_rep    else np.nan,
            'dprime_mean':  np.mean(dprime_per_rep) if dprime_per_rep else np.nan,
            'dprime_std':   np.std(dprime_per_rep)  if dprime_per_rep else np.nan,
        })
    return results

## Section A: Per-ISI Stability (Toy Experiments — Baseline)

Reproduced from `2026-02-19_sigma2-stability.ipynb` as a reference baseline.
k_stimuli set to `ISI + 2`; vary n_experiments in [20, 40, 80, 160].

In [ ]:
EXP_COUNTS = [20, 40, 80, 160]
TEST_ISIS  = [8, 16, 32, 64]
per_isi_results = {}

for isi_val in TEST_ISIS:
    hc_idx = isi_to_hc_idx.get(isi_val)
    if hc_idx is None:
        print(f'ISI {isi_val} not in data — skip'); continue
    k_stim = min(isi_val + 2, len(stimulus_pool))
    print(f'\n=== ISI={isi_val} k={k_stim} human d\'={human_curve[hc_idx]:.4f} ===')
    n_exp_results = {}

    for n_exp in EXP_COUNTS:
        exps = make_toy_experiment_list(
            stimulus_pool, isi=isi_val, n_experiments=n_exp,
            k_stimuli=k_stim, seed=isi_val*1000+n_exp)
        exps = [e for e in exps if e]
        if not exps:
            print(f'  n_exp={n_exp}: EMPTY'); continue
        print(f'  n_exp={n_exp}: {len(exps)} exps, '
              f'avg len {np.mean([len(e) for e in exps]):.1f}')

        res = run_sigma_sweep(
            sigma_name='sigma2', sigma_grid=sigma2_grid,
            fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed, 'sigma2': 0},
            exps_by_isi={isi_val: exps},
            isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
            N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
            metric=metric, X=X, name_to_idx=name_to_idx,
            base_seed=isi_val*100_000_000 + n_exp*1_000_000,
        )
        n_exp_results[n_exp] = res

    per_isi_results[isi_val] = n_exp_results
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, isi_val in zip(axes, TEST_ISIS):
    hc_idx = isi_to_hc_idx.get(isi_val)
    if hc_idx is None or isi_val not in per_isi_results:
        ax.set_title(f'ISI={isi_val} (skip)'); continue
    for n_exp, res in per_isi_results[isi_val].items():
        df = pd.DataFrame(res)
        ax.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N={n_exp}')
        ax.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                        df.mse_mean+df.mse_std, alpha=0.2)
    ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel('MSE')
    ax.set_title(f'ISI={isi_val}  d\'={human_curve[hc_idx]:.2f}  [toy]')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.suptitle(f'Sigma2 per-ISI stability — toy baseline  ({encoder_type}-{layer}-{time_avg_tag})', y=1.03)
plt.tight_layout(); plt.show()

## Section B: Real Sequences — Vary N Sequences per Rep

Each real multi-ISI sequence contains ~5 trials per ISI.
For each MC rep we sample `n_seqs_per_rep` sequences and split
hits by ISI using `infer_trial_isis`.

N = 1 → ~5 trials per ISI per rep (minimum)  
N = 36 → all available sequences (~180 trials per ISI per rep)

In [ ]:
REAL_SEQ_COUNTS = [1, 5, 10, 36]
TARGET_ISIS     = [8, 16, 32, 64]
real_seq_results = {}

for n_seq in REAL_SEQ_COUNTS:
    print(f'\n--- n_seqs_per_rep={n_seq} ---')
    res = run_real_sweep(
        sigma_name='sigma2', sigma_grid=sigma2_grid,
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed, 'sigma2': 0},
        real_exp_list=exp_list,
        n_seqs_per_rep=n_seq,
        target_isis=TARGET_ISIS,
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=200_000_000 + n_seq * 1_000_000,
    )
    real_seq_results[n_seq] = res
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for n_seq, res in real_seq_results.items():
    df = pd.DataFrame(res)
    ax.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N={n_seq}')
    ax.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                    df.mse_mean+df.mse_std, alpha=0.2)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel('MSE')
ax.set_title(f'Sigma2 stability — real seqs  N_MC={N_MC}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for n_seq, res in real_seq_results.items():
    df = pd.DataFrame(res)
    ax.errorbar(df.sigma_value, df.dprime_mean, yerr=df.dprime_std,
                fmt='o-', capsize=3, label=f'N={n_seq}')
mean_human = np.mean([human_curve[isi_to_hc_idx[i]] for i in TARGET_ISIS if i in isi_to_hc_idx])
ax.axhline(mean_human, color='k', ls='--', label=f'Human (mean ISI 8-64)', alpha=0.6)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel("d'")
ax.set_title('d\' vs sigma2 — real seqs')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Section C: ISI Subset Informativeness — Real Sequences

Fixed n_seqs_per_rep=36 (all sequences). Compare all subsets of [8, 16, 32, 64].
Question: can we drop ISI-64 (expensive to toy-generate) and still get a stable fit?

In [ ]:
ISI_SUBSETS = [
    [8],[16],[32],[64],
    [8,16],[8,32],[8,64],[16,32],[16,64],[32,64],
    [8,16,32],[8,16,64],[8,32,64],[16,32,64],
    [8,16,32,64],
]
N_SEQ_FIXED = 36
subset_results_real = {}

for subset in ISI_SUBSETS:
    print(f'--- subset={subset} ---')
    res = run_real_sweep(
        sigma_name='sigma2', sigma_grid=sigma2_grid,
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed, 'sigma2': 0},
        real_exp_list=exp_list,
        n_seqs_per_rep=N_SEQ_FIXED,
        target_isis=subset,
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=sum(subset) * 100_000_000,
    )
    subset_results_real[tuple(subset)] = res
print('Done.')

In [ ]:
# All subsets
plt.figure(figsize=(12, 6))
for subset_key, res in subset_results_real.items():
    df = pd.DataFrame(res)
    plt.plot(df.sigma_value, df.mse_mean, 'o-',
             label=f'{list(subset_key)}', alpha=0.8)
plt.xscale('log'); plt.xlabel(r'$\sigma_2$'); plt.ylabel('Mean MSE')
plt.title(f'Which ISI subsets are most informative? (real seqs, n={N_SEQ_FIXED}, N_MC={N_MC})')
plt.legend(fontsize=6, ncol=3); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Single-ISI only
plt.figure(figsize=(8, 5))
for subset_key, res in subset_results_real.items():
    if len(subset_key) != 1: continue
    df = pd.DataFrame(res)
    plt.plot(df.sigma_value, df.mse_mean, 'o-', label=f'ISI {subset_key[0]}')
    plt.fill_between(df.sigma_value, df.mse_mean-df.mse_std,
                     df.mse_mean+df.mse_std, alpha=0.2)
plt.xscale('log'); plt.xlabel(r'$\sigma_2$'); plt.ylabel('MSE')
plt.title('Single-ISI contribution to sigma2 MSE (real seqs)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Summary

*(Fill in after running)*

**Key findings:**
- Minimum N sequences for stable sigma2 estimate: 
- ISI subsets that are sufficient: 
- Comparison to toy experiment baseline (Section A): 